# Input image format for beamlet image (and backgroud image, optional)
## Take an input image from a pepperpot with a background and output a geometric and normalized emittance value

In [ ]:
import pandas as pd
import numpy as np
import random
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
from itertools import product
from random import shuffle
from math import floor, ceil
import glob
from IPython.display import display
from PIL import Image
from scipy import ndimage
from collections import defaultdict
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
#import cv2
from scipy.stats import gaussian_kde
from matplotlib.patches import Ellipse
from matplotlib import image as mpimg
from collections import deque
from matplotlib.patches import Rectangle

# functions

In [ ]:
## apply the threshold by removing the pixel number that does not exceed the threshold
## using the array that already subtracted the background from the background image
def thresholdsub (arrin,threshold):
    arrout=arrin.copy()
    for iy in range(len(arrin)):
        for ix in range(len(arrin[0])):
            if arrin[iy][ix] > threshold:
                arrout[iy][ix] = arrin[iy][ix]
            else:
                arrout[iy][ix] = 0
    return arrout

In [ ]:
# def apply_threshold(img_array, threshold):
#     thresh_array = img_array - threshold
#     return np.clip(thresh_array, 0, None).astype(np.uint16)

In [ ]:
## raw image with background subtracted
raw_image = 'FE_LEBT_EMS_D0812_20241211_183023_sol=56A_APsout.tiff'
background = 'FE_LEBT_EMS_D0812_20241211_174838_gain5_exp2_bkg.tiff'
raw = Image.open(raw_image).convert('I;16')  
back = Image.open(background).convert('I;16')  
img_array = np.array(raw, dtype=np.int32)
back_array = np.array(back, dtype=np.int32)
data_bkgd = img_array - back_array
data_bkgd = np.array(data_bkgd, dtype=np.int32)
data_bkgd = np.clip(data_bkgd, 0, None).astype(np.uint16)
#fig = plt.figure(figsize = (6, 5))
#plt.imshow(data_bkgd, cmap='jet', vmin=0.0, vmax=data_bkgd.max())
plt.imshow(data_bkgd, cmap='jet')
plt.colorbar(extend='both')
plt.xlabel('x_pixel')
plt.ylabel('y_pixel')
print(np.amax(data_bkgd),np.amin(data_bkgd))

In [ ]:
## raw image with background subtracted and threshold applied
trsd = 1500
data_trsd = thresholdsub(data_bkgd,trsd)
plt.imshow(data_trsd, cmap='jet')
plt.colorbar(extend='both')
plt.xlabel('x_pixel')
plt.ylabel('y_pixel')
print(len(data_trsd),len(data_trsd[0]))

In [ ]:
# Display with the Physical size of the image [mm]
#2560, 70mm
x_range = [-38.8, 38.8]
y_range = [-38.8, 38.8]

# Plot image
plt.imshow(data_trsd, extent=(*x_range, *y_range), cmap='jet', vmin=0.0, vmax=data_trsd.max())
plt.colorbar(extend='both')
plt.xlabel('x [mm]')
plt.ylabel('y [mm]')
print(x_range,y_range)

In [ ]:
## zoom the beamlets area to refine the "threshold"
## first without the threshold removal, only subtraction of the background
x_min_p = -30
x_max_p = 20
y_min_p = -30
y_max_p = 18

# Plot image
plt.imshow(data_bkgd, extent=(*x_range, *y_range), cmap='jet', vmin=0.0, vmax=data_bkgd.max())
plt.colorbar(extend='both')
plt.xlabel('x [mm]')
plt.ylabel('y [mm]')
plt.xlim(x_min_p, x_max_p)
plt.ylim(y_min_p, y_max_p)

In [ ]:
## zoom the beamlets area to refine the "threshold"
## second with the threshold removal, may try different level of threshold

#thresholdtry = 1500
#data_trsd = thresholdsub(data_bkgd,thresholdtry)
plt.imshow(data_trsd, extent=(*x_range, *y_range), cmap='jet', vmin=0.0, vmax=data_trsd.max())
plt.colorbar(extend='both')
plt.xlabel('x [mm]')
plt.ylabel('y [mm]')
plt.xlim(x_min_p, x_max_p)
plt.ylim(y_min_p, y_max_p)


In [ ]:
## Physical size of the image [mm]
# x_range = [-6.6, 7.8]
# y_range = [-8.0, 5.5]

## Plot image
plt.imshow(data_bkgd, extent=(*x_range, *y_range), cmap='jet', vmin=0.0, vmax=data_bkgd.max())

## Get image transformation data to add new layer 
ax = plt.gca()
transx = mpl.transforms.blended_transform_factory(ax.transData, ax.transAxes)
transy = mpl.transforms.blended_transform_factory(ax.transAxes, ax.transData)

win_ratio = 0.2 # scale of the histogram. 1 means fill the plot window.

vx = np.sum(data_bkgd, axis=0) # Projection to X axis
plt.plot(np.linspace(*x_range, data_bkgd.shape[1]), vx/vx.max()*win_ratio, transform=transx, color='orange', lw=1)

vy = np.sum(data_bkgd, axis=1)[::-1] # Projection to Y, reverse order because the image origin is upper left pixel
plt.plot(vy/vy.max()*win_ratio, np.linspace(*y_range, data_bkgd.shape[0]), transform=transy, color='orange', lw=1)

# x_coords = [1, 1, 2, 2, 4,4,5,5,1,1,5]
# y_coords = [1, 5, 5, 1, 1,5,5,1,1,2,2]
# plt.plot(x_coords, y_coords,color='white', lw=1)

# x_min_p = 0
# x_max_p = 5
# y_min_p = -5
# y_max_p = 0

x_min_p = -17
x_max_p = -12
y_min_p = -3
y_max_p = 2

# x_min_p = 3.5
# x_max_p = 4.5
# y_min_p = -2.8
# y_max_p = -1.2

x_start = -17.8
x_stpsiz = 1.5
x_stpnum = 7
x_end = x_start + x_stpsiz*x_stpnum
# x_end = 4.22
# x_stpnum = int((x_end-x_start)/x_stpsiz)+1

y_start = -3.8
y_stpsiz = 1.5
y_stpnum = 8
y_end = y_start + y_stpsiz*y_stpnum
# y_end = -1.2
# y_stpnum = int((y_end-y_start)/y_stpsiz)+1

# grid_x = np.linspace(x_min_p, x_max_p, num=x_stpnum,endpoint=True)
x_yline = np.linspace(x_min_p, x_max_p, 2,endpoint=True)
y_xline = np.linspace(y_min_p, y_max_p, 2,endpoint=True)
grid_x = np.linspace(x_start, x_end, num=x_stpnum,endpoint=True)
grid_y = np.linspace(y_start, y_end, num=y_stpnum,endpoint=True)
for i in range(y_stpnum):
    y_line = (grid_y[i],grid_y[i])
    plt.plot(x_yline, y_line, 'k-')
    plt.plot(x_yline, y_line, 'w-')
for i in range(x_stpnum):
    x_line = (grid_x[i],grid_x[i])
    plt.plot(x_yline, y_line, 'k-')
    plt.plot(x_line, y_xline, 'w-')
    
plt.colorbar(extend='both')
plt.xlabel('x [mm]')
plt.ylabel('y [mm]')
plt.xlim(x_min_p, x_max_p)
plt.ylim(y_min_p, y_max_p)

In [ ]:
def Plot_ImageandGrid(data,xyrange,xy_zoom,xy_grid):
    # data in "npy" format
    # xyrange = ((x_min,xmax),(ymin,ymax)) <-- real size of the image
    # xy_zoom = ((x_zoom_min,x_zoom_max),(y_zoom_min,y_zoom_max)) <-- zoom for plot
    # xy_grid = ((x_start,x_stpsiz,x_stpnum),(y_start,y_stpsiz,y_stpnum)) <-- grid specs
    plt.imshow(data, extent=(*xyrange[0], *xyrange[1]), cmap='flag', vmin=0.0, vmax=data.max())
    # Get image transformation data to add new layer 
    ax = plt.gca()
    transx = mpl.transforms.blended_transform_factory(ax.transData, ax.transAxes)
    transy = mpl.transforms.blended_transform_factory(ax.transAxes, ax.transData)
    
    win_ratio = 0.2 # scale of the histogram. 1 means fill the plot window.
    vx = np.sum(data, axis=0) # Projection to X axis
    plt.plot(np.linspace(*xyrange[0], data.shape[1]), vx/vx.max()*win_ratio, transform=transx, color='black', lw=1)
    vy = np.sum(data, axis=1)[::-1] # Projection to Y, reverse order because the image origin is upper left pixel
    plt.plot(vy/vy.max()*win_ratio, np.linspace(*xyrange[1], data.shape[0]), transform=transy, color='black', lw=1)
    
    x_zoom_min = xy_zoom[0][0]
    x_zoom_max = xy_zoom[0][1]
    y_zoom_min = xy_zoom[1][0]
    y_zoom_max = xy_zoom[1][1]
    
    x_start = xy_grid[0][0]
    x_stpsiz = xy_grid[0][1]
    x_stpnum = xy_grid[0][2]
    x_end = x_start + x_stpsiz*x_stpnum
    y_start = xy_grid[1][0]
    y_stpsiz = xy_grid[1][1]
    y_stpnum = xy_grid[1][2]
    y_end = y_start + y_stpsiz*y_stpnum
    
#     x_yline = np.linspace(x_zoom_min, x_zoom_max, 2,endpoint=True)
#     y_xline = np.linspace(y_zoom_min, y_zoom_max, 2,endpoint=True)
    x_yline = np.linspace(x_start-x_stpsiz, x_end, 2,endpoint=True)
    y_xline = np.linspace(y_start-y_stpsiz, y_end, 2,endpoint=True)
    grid_x = np.linspace(x_start, x_end, num=x_stpnum,endpoint=True)
    grid_y = np.linspace(y_start, y_end, num=y_stpnum,endpoint=True)
    
    for i in range(x_stpnum):
        x_line = (grid_x[i],grid_x[i])
        plt.plot(x_line, y_xline, 'b-')
    for i in range(y_stpnum):
        y_line = (grid_y[i],grid_y[i])
        plt.plot(x_yline, y_line, 'b-')
    
    plt.colorbar(extend='both')
    plt.xlabel('x [mm]')
    plt.ylabel('y [mm]')
    plt.xlim(x_start-x_stpsiz, x_end+x_stpsiz)
    plt.ylim(y_start-y_stpsiz, y_end+y_stpsiz)
#     plt.xlim(x_zoom_min, x_zoom_max)
#     plt.ylim(y_zoom_min, y_zoom_max)

In [ ]:
# x_start = -21.7
# x_stpsiz = 1.5
# x_stpnum = 24

# y_start = -19.7
# y_stpsiz = 1.5
# y_stpnum = 24

xyrange = [x_range,y_range]
xy_zoom = [[1.2,4],[-3.8,-1]]
xy_grid = [[x_start,x_stpsiz,x_stpnum],[y_start,y_stpsiz,y_stpnum]]

Plot_ImageandGrid(data_bkgd,xyrange,xy_zoom,xy_grid)
    # data in "npy" format
    # xyrange = ((x_min,xmax),(ymin,ymax)) <-- real size of the image
    # xy_zoom = ((x_zoom_min,x_zoom_max),(y_zoom_min,y_zoom_max)) <-- zoom for plot
    # xy_grid = ((x_start,x_stpsiz,x_stpnum),(y_start,y_stpsiz,y_stpnum)) <-- grid specs

In [ ]:
Plot_ImageandGrid(data_trsd,xyrange,xy_zoom,xy_grid)

In [ ]:
def Plot_ImageandGrid2(data,xyrange,xy_zoom,xy_grid):
    # data in "npy" format
    # xyrange = ((x_min,xmax),(ymin,ymax)) <-- real size of the image
    # xy_zoom = ((x_zoom_min,x_zoom_max),(y_zoom_min,y_zoom_max)) <-- zoom for plot
    # xy_grid = ((x_start,x_stpsiz,x_stpnum),(y_start,y_stpsiz,y_stpnum)) <-- grid specs
    plt.imshow(data, extent=(*xyrange[0], *xyrange[1]), cmap='flag', vmin=0.0, vmax=data.max())
    # Get image transformation data to add new layer 
    ax = plt.gca()
    transx = mpl.transforms.blended_transform_factory(ax.transData, ax.transAxes)
    transy = mpl.transforms.blended_transform_factory(ax.transAxes, ax.transData)
    
    win_ratio = 0.2 # scale of the histogram. 1 means fill the plot window.
    vx = np.sum(data, axis=0) # Projection to X axis
    plt.plot(np.linspace(*xyrange[0], data.shape[1]), vx/vx.max()*win_ratio, transform=transx, color='black', lw=1)
    vy = np.sum(data, axis=1)[::-1] # Projection to Y, reverse order because the image origin is upper left pixel
    plt.plot(vy/vy.max()*win_ratio, np.linspace(*xyrange[1], data.shape[0]), transform=transy, color='black', lw=1)
    
    x_zoom_min = xy_zoom[0][0]
    x_zoom_max = xy_zoom[0][1]
    y_zoom_min = xy_zoom[1][0]
    y_zoom_max = xy_zoom[1][1]
    
    x_start = xy_grid[0][0]
    x_stpsiz = xy_grid[0][1]
    x_stpnum = xy_grid[0][2]
    x_end = x_start + x_stpsiz*x_stpnum
    y_start = xy_grid[1][0]
    y_stpsiz = xy_grid[1][1]
    y_stpnum = xy_grid[1][2]
    y_end = y_start + y_stpsiz*y_stpnum
    
#     x_yline = np.linspace(x_zoom_min, x_zoom_max, 2,endpoint=True)
#     y_xline = np.linspace(y_zoom_min, y_zoom_max, 2,endpoint=True)
    x_yline = np.linspace(x_start-x_stpsiz, x_end, 2,endpoint=True)
    y_xline = np.linspace(y_start-y_stpsiz, y_end, 2,endpoint=True)
    grid_x = np.linspace(x_start, x_end, num=x_stpnum,endpoint=True)
    grid_y = np.linspace(y_start, y_end, num=y_stpnum,endpoint=True)
    
    for i in range(x_stpnum):
        x_line = (grid_x[i],grid_x[i])
        plt.plot(x_line, y_xline, 'b-', linewidth = 0.5)
    for i in range(y_stpnum):
        y_line = (grid_y[i],grid_y[i])
        plt.plot(x_yline, y_line, 'b-', linewidth = 0.5)
    
    plt.colorbar(extend='both')
    plt.xlabel('x [mm]')
    plt.ylabel('y [mm]')
    plt.xlim(x_start-x_stpsiz, x_end+x_stpsiz)
    plt.ylim(y_start-y_stpsiz, y_end+y_stpsiz)
    plt.xlim(x_zoom_min, x_zoom_max)
    plt.ylim(y_zoom_min, y_zoom_max)

In [ ]:
# x_start = -22.7
# x_end = 16.8
# x_stpnum = 24

# y_start = -20.3
# y_end = 15.8
# y_stpnum = 23
# xyrange = [x_range,y_range]
# xy_zoom = [[-23, 17],[-21, 17]]
# #xy_zoom = [[10, 20],[-10, 0]]
# xy_grid = [[x_start,x_stpsiz,x_stpnum],[y_start,y_stpsiz,y_stpnum]]
# Plot_ImageandGrid2(data_trsd,xyrange,xy_zoom,xy_grid)

In [ ]:
## Split the array with x, y, p into 3 seperate arrays for masking purposes
def extract_xyp(real):
    flattened = np.array(real.tolist(), dtype = float).reshape(-1, 3)
    x = flattened[:, 0]
    y = flattened[:, 1]
    p = flattened[:, 2]
    return x, y, p
## Function for grouping beamlets based on manually determined boundaries
def grouping(real_array, Nx, xmin, xmax, dx, Ny, ymin, ymax, dy):
    x, y, p = extract_xyp(real_array)
    x_edges = np.linspace(xmin, xmax, num = Nx, endpoint = True)
    y_edges = np.linspace(ymin, ymax, num = Ny, endpoint = True)
    x_edge_grid, y_edge_grid = np.meshgrid(x_edges, y_edges)
    width = (xmax-xmin)/Nx
    height = (ymax-ymin)/Ny
    #create the boundaries using np.stack because we can get those values from the inputs
    boundaries = np.stack((x_edge_grid, y_edge_grid, np.full_like(x_edge_grid, width), np.full_like(y_edge_grid, height)), axis = -1) #np.full_like fills an array with the same value to be the same size as the first argument
    grouped_points = np.empty((Ny, Nx), dtype = object) #need to set dtype to object so it can store lists
    grouped_points[:, :] = [[[] for _ in range(Nx)] for _ in range(Ny)]
    #reshape x, y, and p for masking 
    x = x[:, np.newaxis, np.newaxis]
    y = y[:, np.newaxis, np.newaxis]
    p = p[:, np.newaxis, np.newaxis]
    #Extract all the values from boundaries
    x_min = boundaries[..., 0]
    y_min = boundaries[..., 1]
    w = boundaries[..., 2]
    h = boundaries[..., 3]
    # Broadcast boundary arrays to shape (1, Ny, Nx)
    x_min = x_min[np.newaxis, :, :]
    y_min = y_min[np.newaxis, :, :]
    w = w[np.newaxis, :, :]
    h = h[np.newaxis, :, :]
    # Mask of shape (N, Ny, Nx): True if point falls inside the corresponding box
    in_bounds = (
        (x >= x_min) & (x < x_min + w) &
        (y >= y_min) & (y < y_min + h) &
        (p > 0))
    point_indices = np.where(in_bounds)
    for point_idx, j, i in zip(*point_indices):
        xi = float(x[point_idx])
        yi = float(y[point_idx])
        pi = float(p[point_idx])
        grouped_points[j][i].append((xi, yi, pi))
    return grouped_points, boundaries

In [ ]:
## Creates a grid of values that for the entire size of the image has corresponding x, y, p for each pixel, needs range in x and y direction
def convert_to_real_range(raw_value, x_range, y_range):
    height, width = raw_value.shape
    real_coords = np.empty((height, width), dtype = object)
    x_values = np.linspace(x_range[0], x_range[1], width)
    y_values = np.linspace(y_range[1], y_range[0], height)
    X, Y = np.meshgrid(x_values, y_values)
    full_grid = np.stack((X, Y, raw_value), axis=-1)
    return full_grid

In [ ]:
def create_screen(groups, index, distance, hole_position): #distance refers to distance between holes in the screen, index refers to the index in groups of which beamlet has an associated hole
    #find the height and width of groups to match the screen to it
    height = len(groups)
    width = len(groups[0])
    x_distance = distance[0]
    y_distance = distance[1]
    hole_rows = [[[] for _ in range(width)] for _ in range(height)]
    hole_rows[index[0]][index[1]] = hole_position
    for i in range(len(hole_rows)):
        for j in range(len(hole_rows[0])):
            if [i, j] != index:
                diff_j = j - index[1]
                diff_i = i - index[0] 
                hole = (hole_position[0] + diff_j * x_distance, hole_position[1] + diff_i * y_distance) #remember x is j and y is i
                hole_rows[i][j] = hole
    return hole_rows

In [ ]:
def calc_angles(holes, beamlets, L, centers): 
    total = []
    for i in range(len(beamlets)):
        for j in range(len(beamlets[i])):
            if len(beamlets[i][j]) > 0:
                #print(beamlets[i][j])
                print(holes[i][j])
                print(centers[i][j])
                for k in range(len(beamlets[i][j])):
                    angle_x = (beamlets[i][j][k][0] - holes[i][j][0]) / L 
                    angle_y = (beamlets[i][j][k][1] - holes[i][j][1]) / L
                    temp = np.concatenate([beamlets[i][j][k], [angle_x, angle_y]])
                    total.append(temp)
    return total 

In [ ]:
def plot_boundaries(boundaries, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize = (12,10))

    Ny, Nx, _ = boundaries.shape

    for j in range(Ny):
        for i in range(Nx):
            x_min, y_min, width, height = boundaries[j, i]
            rect = Rectangle((x_min, y_min), width, height,
                             edgecolor='red', facecolor='none', linewidth=1)
            ax.add_patch(rect)

    ax.set_aspect('equal')
    ax.autoscale()
    return ax

In [ ]:
x_start = -22.3
x_end = 18
x_stpnum = 26 ##should be equal to N+1 beamlets

y_start = -20.3
y_end = 15.8
y_stpnum = 23

In [ ]:
x_yline = np.linspace(x_start, x_end, 2,endpoint=True)
y_xline = np.linspace(y_start, y_end, 2,endpoint=True)
grid_x = np.linspace(x_start, x_end, num=x_stpnum,endpoint=True)
grid_y = np.linspace(y_start, y_end, num=y_stpnum,endpoint=True)

In [ ]:
# xy_zoom = [[-20, 20],[-25, 20]]
# plt.figure(figsize = (12, 10))
# # ax = plot_boundaries(bounds)
# plt.scatter(x, y, c = p, cmap = 'flag', s = 1)
# # plt.scatter(centers_x, centers_y, color = 'green')
# # plt.scatter(hole_x, hole_y, color = 'purple')
# for i in range(x_stpnum):
#     x_line = (grid_x[i],grid_x[i])
#     plt.plot(x_line, y_xline, 'b-')
# for i in range(y_stpnum):
#     y_line = (grid_y[i],grid_y[i])
#     plt.plot(x_yline, y_line, 'b-')
# #plt.colorbar()
# # plt.xlim(xy_zoom[0])
# # plt.ylim(xy_zoom[1])
# plt.show()

In [ ]:
real_data_trsd = convert_to_real_range(data_trsd, x_range, y_range)

In [ ]:
group, bounds = grouping(real_data_trsd,  x_stpnum, x_start, x_end, 0, y_stpnum, y_start, y_end, 0)

In [ ]:
## Convert internal lists to arrays and put conditions to be deemed beamlet
min_group_size = 5
a_group = np.empty((len(group), len(group[0])), dtype = object)
for n in range(len(group)):
    for m in range(len(group[n])):
        if len(group[n][m]) > min_group_size:
            a_group[n][m] = np.array(group[n][m])
        else:
            a_group[n][m] = np.array([])

In [ ]:
centers = np.empty((len(a_group), len(a_group[0])), dtype = object)
for i in range(len(a_group)):
    for j in range(len(a_group[i])):
        if len(a_group[i][j]) > 0:
            x_center = np.average(a_group[i][j][:, 0], weights = a_group[i][j][:, 2])
            y_center = np.average(a_group[i][j][:, 1], weights = a_group[i][j][:, 2])
            temp = [x_center, y_center]
            centers[i][j] = temp
        else:
            centers[i][j] = 'None'

In [ ]:
x = []
y = []
p = []
for i in range(len(a_group)):
    for j in range(len(a_group[i])):
        for k in range(len(a_group[i][j])):
            x.append(a_group[i][j][k][0])
            y.append(a_group[i][j][k][1])
            p.append(a_group[i][j][k][2])

In [ ]:
## Choose the beamlet index that has a known associated hole in the screen
index = [14, 13]
hole_position = centers[index[0]][index[1]]
x_stpsiz = (x_end - x_start)/(x_stpnum-1)
y_stpsiz = (y_end - y_start)/(y_stpnum-1)
distance = [x_stpsiz, y_stpsiz]
hole_rows = create_screen(a_group, index, distance, hole_position)

In [ ]:
centers_x = []
centers_y = []
hole_x = []
hole_y = []
for i in range(len(hole_rows)):
    for j in range(len(hole_rows[i])):
        if centers[i][j] != 'None':
            centers_x.append(centers[i][j][0])
            centers_y.append(centers[i][j][1])
            hole_x.append(hole_rows[i][j][0])
            hole_y.append(hole_rows[i][j][1])

In [ ]:
# x_start = xy_grid[0][0]
# x_stpsiz = xy_grid[0][1]
# x_stpnum = xy_grid[0][2]
# x_end = x_start + x_stpsiz*x_stpnum
# y_start = xy_grid[1][0]
# y_stpsiz = xy_grid[1][1]
# y_stpnum = xy_grid[1][2]
# y_end = y_start + y_stpsiz*y_stpnum

In [ ]:
# x_start = -21.9
# x_stpsiz = 1.51
# x_stpnum = 24

# y_start = -19.9
# y_stpsiz = 1.55
# y_stpnum = 24
# x_start = -21.99
# x_stpsiz = 1.461
# x_stpnum = 23

# y_start = -19.9
# y_stpsiz = 1.485
# y_stpnum = 23

# xyrange = [x_range,y_range]
# xy_zoom = [[-22, 14.5],[-20, 17.5]]
# xy_grid = [[x_start,x_stpsiz,x_stpnum],[y_start,y_stpsiz,y_stpnum]]

# x_yline = np.linspace(x_start, x_end, 2,endpoint=True)
# y_xline = np.linspace(y_start, y_end, 2,endpoint=True)
# grid_x = np.linspace(x_start, x_end, num=x_stpnum,endpoint=True)
# grid_y = np.linspace(y_start, y_end, num=y_stpnum,endpoint=True)

In [ ]:
plt.figure(figsize = (12, 10))
ax = plot_boundaries(bounds)
plt.scatter(x, y, c = p, cmap = 'flag', s = 1)
plt.scatter(centers_x, centers_y, color = 'green')
plt.scatter(hole_x, hole_y, color = 'purple')
for i in range(x_stpnum):
    x_line = (grid_x[i],grid_x[i])
    plt.plot(x_line, y_xline, 'b-')
for i in range(y_stpnum):
    y_line = (grid_y[i],grid_y[i])
    plt.plot(x_yline, y_line, 'b-')
#plt.colorbar()
plt.show()

In [ ]:
L = 41 #length in millimeters ReA
# L = 30 #length in millimeter FRIB
total = calc_angles(hole_rows, a_group, L, centers)

In [ ]:
x_cal = []
y_cal = []
p_cal = []
xp_cal = []
yp_cal = []
for i in range(len(total)):
    if total[i][2] > 0:
        x_cal.append(total[i][0])
        y_cal.append(total[i][1])
        p_cal.append(total[i][2])
        xp_cal.append(total[i][3])
        yp_cal.append(total[i][4])

In [ ]:
fig = plt.figure(figsize = (12, 12))
ax1 = fig.add_subplot(2, 2, 1)
ax1.scatter(x_cal, xp_cal, s = 0.5, color = 'blue')
ax1.set_xlabel('x (mm)')
ax1.set_ylabel('x angle (rad)')
ax1.grid()
# for i in range(x_stpnum):
#     x_line = (grid_x[i],grid_x[i])
#     ax1.plot(x_line, y_xline, 'b-')
ax2 = fig.add_subplot(2, 2, 2)
ax2.scatter(y_cal, yp_cal, s = 0.5, color = 'red')
ax2.set_xlabel('y (mm)')
ax2.set_ylabel('y angle (rad)')
ax3 = fig.add_subplot(2, 2, 3)
ax3.scatter(x_cal, y_cal, s= 0.5, color = 'blue')
ax3.set_xlabel('x (mm)')
ax3.set_ylabel('y (mm)')
ax4 = fig.add_subplot(2, 2, 4)
ax4.scatter(xp_cal, yp_cal, s = 0.5, color = 'red')
ax4.set_xlabel('x angle (rad)')
ax4.set_ylabel('y angle (rad)')
plt.show()

In [ ]:
fig = plt.figure(figsize = (12, 12))
ax1 = fig.add_subplot(2, 2, 1)
ax1.scatter(x_cal, yp_cal, s = 1, color = 'blue')
ax1.set_xlabel('x (mm)')
ax1.set_ylabel('y angle (rad)')
ax2 = fig.add_subplot(2, 2, 2)
ax2.scatter(y_cal, xp_cal, s = 1, color = 'red')
ax2.set_xlabel('y (mm)')
ax2.set_ylabel('x angle (rad)')
ax3 = fig.add_subplot(2, 2, 3)
ax3.scatter(x_cal, y_cal, s= 1, color = 'blue')
ax3.set_xlabel('x (mm)')
ax3.set_ylabel('y (mm)')
ax4 = fig.add_subplot(2, 2, 4)
ax4.scatter(xp_cal, yp_cal, s = 1, color = 'red')
ax4.set_xlabel('x angle (rad)')
ax4.set_ylabel('y angle (rad)')
plt.show()

# Calculation Functions

In [ ]:
#beam emittance function
def beam_emittance(values_1, values_2, weights):
    mean_1 = np.average(values_1, weights = weights)
    mean_2 = np.average(values_2, weights = weights)
    mean_squared1 = np.average(np.square(values_1 - mean_1), weights = weights)
    mean_squared2 = np.average(np.square(values_2 - mean_2), weights = weights)
    covariance = np.average((values_1-mean_1)*(values_2-mean_2), weights = weights)
    emittance = np.sqrt(mean_squared1 * mean_squared2 - covariance**2)
    beta = mean_squared1/emittance
    gamma = mean_squared2/emittance
    alfa = -1*covariance/emittance
    parameters = [alfa, beta, gamma]
    return emittance, parameters

In [ ]:
emit_x, param_x = beam_emittance(x_cal, xp_cal, p_cal)
print('Emittance_x =',f'{emit_x*1000:.3f}', '(mm-mrad)')
print('alfa_x =',f'{param_x[0]:.3f}', '  beta_x =', f'{param_x[1]/1000:.3f}', 'm/rad')
emit_y, param_y = beam_emittance(y_cal, yp_cal, p_cal)
print('Emittance_y =',f'{emit_y*1000:.3f}', '(mm-mrad)')
print('alfa_y =',f'{param_y[0]:.3f}', '  beta_y =', f'{param_y[1]/1000:.3f}', 'm/rad')

In [ ]:
def calc_4d_weighted(x, y, xp, yp, weights):
    data = np.stack([x, y, xp, yp], axis=0)  # shape: (4, N)
    weights = np.array(weights, dtype=np.float64)
    weights /= np.sum(weights)
    row_sums = np.sum(data * weights[np.newaxis, :], axis=1, keepdims=True)
    centered = data - row_sums
    weighted_centered = centered * weights[np.newaxis, :]
    covariance_matrix = weighted_centered @ centered.T
    determinant = np.linalg.det(covariance_matrix)
    emit_4d = np.sqrt(determinant)
    return emit_4d

In [ ]:
emit_4d = calc_4d_weighted(x_cal, y_cal, xp_cal, yp_cal, p_cal)
print('4D_Emittance =',f'{emit_4d*1000000:.3e}', '(mm-mrad)^2')

In [ ]:
def plot_3D(x, y, z, xlabel, ylabel, zlabel, title):
    x = np.array(x)
    y = np.array(y)
    z = np.array(z)
    X, Y = np.meshgrid(np.linspace(min(x), max(x), len(np.unique(x))), 
                       np.linspace(min(y), max(y), len(np.unique(y))))
    Z = np.zeros(X.shape)
    for i in range(len(x)):
        xi = np.searchsorted(X[0], x[i])
        yi = np.searchsorted(Y[:, 0], y[i])
        Z[yi, xi] = z[i]
    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z)])
    fig.update_traces(contours_z=dict(show=True, usecolormap=True,
                                      highlightcolor="limegreen", project_z=True))
    fig.update_layout(
        title=title, 
        scene=dict(
            xaxis_title=xlabel,
            yaxis_title=ylabel,
            zaxis_title=zlabel,
            camera=dict(eye=dict(x=1.87, y=0.88, z=-0.64))
        ),
        width=800, 
        height=800,
        margin=dict(l=65, r=50, b=65, t=90)
    )
    fig.show()

In [ ]:
plot_3D(x_cal, xp_cal, p_cal, 'x (mm)', 'xp (rad)', 'p', '3D Contour Plot')